## Stacked barplots

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Load your actual data
# df = pd.read_csv('results.csv')

# --- Helper Function for Stacked Bars ---
def plot_cost_comparison(df_subset, x_label_col, title, save_path):
    """
    Creates a stacked bar chart specifically for cloud cost components.
    """
    # Select only the cost columns for the stack
    cost_cols = ['storage_chf', 'requests_chf', 'transfer_chf']
    
    # Set the X-axis labels
    df_plot = df_subset.set_index(x_label_col)[cost_cols]
    
    # Create the plot
    ax = df_plot.plot(
        kind='bar', 
        stacked=True, 
        figsize=(10, 6), 
        color=['#3498db', '#e67e22', '#2ecc71'] # Blue, Orange, Green
    )
    
    plt.title(title, fontsize=14, fontweight='bold')
    plt.ylabel('Total Cost (CHF)', fontsize=12)
    plt.xlabel('Configuration', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.legend(title="Cost Breakdown", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()

# --- COMPARISON 1: CSV vs Parquet (Across S, M, L) ---
# Filter for baseline files (single file) to compare formats fairly
csv_vs_pq = df[
    (df['n_files'] == 1) & 
    (df['variant'].isin(['csv', 'parquet']))
].copy()
csv_vs_pq['label_variant'] = csv_vs_pq['label'] + " (" + csv_vs_pq['variant'] + ")"

plot_cost_comparison(
    csv_vs_pq, 
    'label_variant', 
    'Cost Comparison: CSV vs Parquet (S, M, L)', 
    'comparison_format.png'
)

# --- COMPARISON 2: Snappy vs ZSTD (Fixed Size: Medium) ---
# Filter for Medium size and Parquet to compare compression only
compression_df = df[
    (df['label'] == 'M') & 
    (df['variant'] == 'parquet') & 
    (df['n_files'] == 1)
].copy()

plot_cost_comparison(
    compression_df, 
    'compression', 
    'Cost Comparison: Snappy vs ZSTD (Medium Size)', 
    'comparison_compression.png'
)

# --- COMPARISON 3: Small Files vs Compact (Fixed Size: Medium) ---
# Compare 100k line segments vs single file
files_df = df[
    (df['label'] == 'M') & 
    (df['variant'] == 'parquet') & 
    (df['compression'] == 'snappy')
].copy()
files_df['sizing'] = files_df['n_files'].apply(lambda x: "Small (100k)" if x > 1 else "Compact")

plot_cost_comparison(
    files_df, 
    'sizing', 
    'Cost Comparison: File Segmentation (Medium Size)', 
    'comparison_segmentation.png'
)